# 🧠 BharatBench — Deep Learning Models: CNN & ConvLSTM

**Notebook:** `3_CNN.ipynb`  
**Dataset:** BharatBench (IMDAA reanalysis, 1990–2020, 1.08° resolution, 32×32 grid)  
**Paper:** *Preparing benchmarks for data-driven weather forecasting system over India*  
**Dataset DOI / Download:** [Kaggle — maslab/bharatbench](https://www.kaggle.com/datasets/maslab/bharatbench)  
**Code Repository:** [GitHub — MASLABnitrkl/BharatBench](https://github.com/MASLABnitrkl/BharatBench)

---

## Purpose

This notebook trains and evaluates two deep learning architectures on the BharatBench dataset:

| Model | Key idea | Input shape |
|---|---|---|
| **CNN** (encoder-decoder) | Learns spatial patterns via convolutional feature maps; encodes the input field into a compressed latent space, then decodes back to full resolution | `(batch, 32, 32, 1)` |
| **ConvLSTM** (encoder-decoder) | Combines spatial convolution with LSTM memory cells; designed for spatio-temporal sequences | `(batch, 1, 32, 32, 1)` |

Both models follow an **encoder–decoder** design: spatial resolution is progressively halved through the encoder (pooling) and then restored through the decoder (upsampling). This architecture encourages the model to learn compressed, high-level representations of the atmospheric state before reconstructing the target field.

### Why encoder-decoder?

Standard CNNs applied to images preserve spatial resolution throughout. The encoder-decoder bottleneck forces the network to capture the dominant large-scale patterns of atmospheric circulation (e.g., the position and intensity of pressure systems) rather than memorising local pixel patterns — which is exactly the generalisation needed for weather forecasting.

---

## Variables evaluated

| Short name | Variable | Level |
|---|---|---|
| `HGT_prl` | Geopotential Height | 500 hPa |
| `TMP_prl` | Temperature | 850 hPa |
| `TMP_2m` | Temperature | 2 m surface |
| `APCP_sfc` | 6-hourly Accumulated Precipitation | Surface |

> 💡 The notebook is currently configured to train on **T850 (`TMP_prl`)** at a **5-day (20 time step)** lead time. To switch variables, update `ds` in Section 3 and the denormalisation variable name (`std.TMP_prl`, `mean.TMP_prl`) in the evaluation cells.

---

## Notebook structure

1. [Setup & Imports](#1-setup)
2. [Load Dataset](#2-dataset)
3. [Variable Selection & Data Splits](#3-splits)
4. [Data Pipeline](#4-pipeline)
5. [Evaluation Metrics](#5-metrics)
6. [Part A — CNN Model](#6-cnn)
   - [6.1 Architecture](#6-1-arch)
   - [6.2 Build & Compile](#6-2-build)
   - [6.3 Callbacks](#6-3-callbacks)
   - [6.4 Training](#6-4-train)
   - [6.5 Save & Plot History](#6-5-save)
   - [6.6 Architecture Visualisation](#6-6-viz)
   - [6.7 Evaluation](#6-7-eval)
7. [Part B — ConvLSTM Model](#7-convlstm)
   - [7.1 Reshape for ConvLSTM](#7-1-reshape)
   - [7.2 Architecture](#7-2-arch)
   - [7.3 Build & Compile](#7-3-build)
   - [7.4 Callbacks & Training](#7-4-train)
   - [7.5 Save & Load](#7-5-save)
   - [7.6 Evaluation](#7-6-eval)

---
<a id="1-setup"></a>
## 1. Setup & Imports

Core scientific Python libraries and TensorFlow/Keras are imported here. The TensorFlow import also initialises GPU detection — any available GPU will be used automatically for training.

> **Environment check:** If any import fails, install the missing packages with:
> ```bash
> pip install numpy xarray matplotlib tensorflow
> ```
> TensorFlow 2.x is required. GPU support requires the appropriate CUDA and cuDNN versions matching your TF installation.

In [ ]:
# Libraries

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import tensorflow.keras as keras
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')

---
<a id="2-dataset"></a>
## 2. Load Dataset

The full BharatBench dataset is loaded as a single NetCDF file containing all variables for 1990–2020.

> ⚠️ **Update the file path below** to match the location of your downloaded dataset.  
> The dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/maslab/bharatbench).

**Dataset specifications:**

| Property | Value |
|---|---|
| Spatial domain | 5°N – 40°N, 65°E – 100°E |
| Spatial resolution | 1.08° (32 × 32 grid) |
| Temporal resolution | 6-hourly (00, 06, 12, 18 UTC) |
| Period | 1990–2020 |

In [ ]:
data =  xr.open_dataset(r"G:/IMDAA_Regrid_1.08_1990_2022/IMDAA_merged_1.08_1990_2020.nc")
data

---
<a id="3-splits"></a>
## 3. Variable Selection & Data Splits

### 3.1 Available variables

All four BharatBench target variables are listed for reference:

In [ ]:
var_name = ['HGT_prl', 'TMP_prl', 'TMP_2m', 'APCP_sfc'] # [H500, T850, T2m, TP6h]

### 3.2 Select the target variable

The model is trained on a **single variable at a time**. Here, `TMP_prl` (temperature at 850 hPa) is selected and converted to an `xr.Dataset` for compatibility with the pipeline function.

> 🔁 **To train on a different variable**, replace `'TMP_prl'` with any variable name from `var_name` above, and update the denormalisation variable names in the evaluation cells (Sections 6.7 and 7.6) accordingly.

In [ ]:
ds = data['TMP_prl'] 
ds = ds.to_dataset()
ds

### 3.3 Dataset splits and lead time

The dataset uses three non-overlapping temporal splits. The **validation set (2018)** is held separate from training so that `EarlyStopping` can monitor generalisation during training without contaminating the final test evaluation.

| Split | Period | Purpose |
|---|---|---|
| **Training** | 1990–2017 (28 years) | Fit model weights |
| **Validation** | 2018 (1 year) | Monitor overfitting; drive `EarlyStopping` |
| **Test** | 2019–2020 (2 years) | Final skill evaluation (held out) |

`lead_time_steps = 20` corresponds to **5 days ahead** (20 × 6-hour steps).  
To evaluate at **3 days ahead**, change this to `lead_time_steps = 12`.

In [ ]:
# training dataset selection
train_years = slice('1990', '2017')
# validation dataset selection (this dataset helps with overfitting)
valid_years = slice('2018', '2018')
# test dataset selection
test_years = slice('2019', '2020')
# prediction days ahead
lead_time_steps = 20 # consider the number of observations per day

---
<a id="4-pipeline"></a>
## 4. Data Pipeline

### 4.1 Pipeline function — `get_train_valid_test_dataset`

This function performs three operations:

**Step 1 — Temporal split:** Divides the dataset into train, validation, and test periods.

**Step 2 — Normalisation:** Computes the global mean and standard deviation from the training data alone, then applies z-score normalisation to all three splits:
$$x_{\text{norm}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

Note that a global scalar mean/std (across all times and grid points) is used rather than a per-pixel mean/std. This preserves the spatial variability of the field — regions with systematically higher or lower values retain their relative differences, which is important for capturing large-scale atmospheric patterns.

**Step 3 — Lead-time shift:** Constructs input–output pairs by shifting the time axis:
- **X (input):** time steps $t_0 \ldots t_{N-\Delta t}$  
- **Y (target):** time steps $t_{\Delta t} \ldots t_N$

A trailing `[..., None]` adds a channel dimension, giving arrays of shape `(time, lat, lon, 1)` expected by `Conv2D`.

In [ ]:
def get_train_valid_test_dataset(lead_steps, Data_array):
  # Split train, valid and test dataset
  train_data = Data_array.sel(time=train_years)
  valid_data = Data_array.sel(time=valid_years)
  test_data = Data_array.sel(time=test_years)

  # Normalize the data using the mean and standard deviation of the training data
  # mean = train_data.mean(dim = "time")
  # std = train_data.std(dim = "time")
  
  mean = train_data.mean()
  std = train_data.std()

  train_data = (train_data - mean) / std
  valid_data = (valid_data - mean) / std
  test_data = (test_data - mean) / std

  # Create inputs and outputs that are shifted by lead_steps
  X_train = train_data[list(Data_array)[0]].isel(time=slice(None, -lead_steps)).values[..., None]
  Y_train = train_data[list(Data_array)[0]].isel(time=slice(lead_steps, None)).values[..., None]
  X_valid = valid_data[list(Data_array)[0]].isel(time=slice(None, -lead_steps)).values[..., None]
  Y_valid = valid_data[list(Data_array)[0]].isel(time=slice(lead_steps, None)).values[..., None]
  X_test = test_data[list(Data_array)[0]].isel(time=slice(None, -lead_steps)).values[..., None]
  Y_test = test_data[list(Data_array)[0]].isel(time=slice(lead_steps, None)).values[..., None]
  return X_train, Y_train, X_valid, Y_valid, X_test, Y_test, mean, std

### 4.2 Run the pipeline

Execute the pipeline for the selected variable and lead time. The returned `mean` and `std` scalars are saved for denormalisation at evaluation time.

In [ ]:
X_train, Y_train, X_valid, Y_valid, X_test, Y_test, mean, std = get_train_valid_test_dataset(lead_time_steps, ds)

Inspect the training-set global mean value for the selected variable:

In [ ]:
mean

Inspect the training-set global standard deviation — this value is used to scale model predictions back to physical units:

In [ ]:
std

### 4.3 Verify array shapes

Confirm that all arrays have the expected shape `(time, lat=32, lon=32, channels=1)`. The number of time steps in X and Y will be `original_steps - lead_time_steps`.

In [ ]:
print(X_train.shape)
print(Y_train.shape)
print(X_valid.shape)
print(Y_valid.shape)
print(X_test.shape)
print(Y_test.shape)

---
<a id="5-metrics"></a>
## 5. Evaluation Metrics

The same three verification metrics used throughout BharatBench are defined here. These are identical to those in the previous notebooks and are reproduced to keep this notebook self-contained.

### 5.1 Root Mean Square Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{N_{\text{pred}}} \sum_{i} \frac{1}{N_{\text{lat}} N_{\text{lon}}} \sum_{j,k} (f_{i,j,k} - t_{i,j,k})^2}$$

The **primary ranking metric** for BharatBench. Reported in the original physical units of each variable (metres for Z500, Kelvin for temperatures, kg/m² for precipitation).

In [ ]:
def compute_rmse(prediction, actual,  mean_dims = ('time', 'latitude', 'longitude')):
  error = prediction - actual
  rmse = np.sqrt(((error)**2 ).mean(mean_dims))
  return rmse

### 5.2 Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{N_{\text{pred}}} \sum_{i} \frac{1}{N_{\text{lat}} N_{\text{lon}}} \sum_{j,k} |f_{i,j,k} - t_{i,j,k}|$$

In [ ]:
def compute_mae(prediction, actual, mean_dims = ('time', 'latitude', 'longitude')):
    error = prediction - actual
    mae = np.abs(error).mean(mean_dims)
    return mae

### 5.3 Anomaly Correlation Coefficient (ACC)

$$\text{ACC} = \frac{\sum_{i,j,k} f'_{i,j,k}\, t'_{i,j,k}}{\sqrt{\sum_{i,j,k} f'^2_{i,j,k} \cdot \sum_{i,j,k} t'^2_{i,j,k}}}$$

where primed variables denote departures from the climatological mean.

| ACC range | Interpretation |
|---|---|
| > 0.8 | Highly skillful forecast |
| ≈ 0.6 | Useful forecast |
| ≈ 0.5 | Comparable to climatological mean |
| < 0.3 | Poor skill |

In [ ]:
def compute_acc(prediction, actual):
    clim = actual.mean('time')
    try:
        t = np.intersect1d(prediction.time, actual.time)
        pred_anomaly = prediction.sel(time=t) - clim
    except AttributeError:
        t = actual.time.values
        pred_anomaly = prediction - clim
    act_anomaly = actual.sel(time=t) - clim
    
    pred_norm = pred_anomaly - pred_anomaly.mean()
    act_norm = act_anomaly - act_anomaly.mean()

    acc = (
            np.sum(pred_norm * act_norm) /
            np.sqrt(
                np.sum(pred_norm ** 2) * np.sum(act_norm ** 2)
            )
    )
    return acc

---
<a id="6-cnn"></a>
## Part A — CNN Encoder-Decoder Model

The Convolutional Neural Network (CNN) used here follows a classic **encoder-decoder** (also called U-Net-style) architecture. It is the primary deep learning baseline in BharatBench.

### Why CNNs for weather forecasting?

- **Translation equivariance:** A weather pattern recognised in one region of the domain will be recognised identically if it shifts location — a useful inductive bias for atmospheric dynamics.
- **Hierarchical feature learning:** Shallow layers capture local gradients and fronts; deeper layers capture large-scale circulation patterns.
- **Computational efficiency:** Conv2D operations on a 32 × 32 grid are extremely fast compared to fully-connected alternatives.

### Encoder-decoder architecture overview

```
Input (32×32×1)
│
├─ Encoder (×4 blocks)
│   Conv2D(32, 5×5, swish) → MaxPooling2D(2×2)
│   Spatial resolution: 32 → 16 → 8 → 4 → 2
│
├─ Dropout(0.2)   ← regularisation bottleneck
│
├─ Decoder (×4 blocks)
│   Conv2D(32, 5×5, swish) → UpSampling2D(2×2)
│   Spatial resolution: 2 → 4 → 8 → 16 → 32
│
└─ Output Conv2D(1, 5×5, linear)
   Shape: (32×32×1) — one predicted field
```

> **Activation — Swish:** The Swish activation $f(x) = x \cdot \sigma(x)$ is used throughout. Swish is smooth and non-monotonic, and has been shown to outperform ReLU for deep networks in practice (Ramachandran et al., 2017).  
> **Output layer — linear (no activation):** Since this is a regression task (predicting a continuous field), no activation is applied at the output.

### Additional TensorFlow/Keras imports

Extended layer imports required for building both the CNN and ConvLSTM architectures:

In [ ]:
import tensorflow.keras as keras
from tensorflow.keras import Model, Sequential

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import MeanAbsoluteError

from tensorflow.keras.layers import Dense, Conv1D, LSTM, Lambda, Reshape, RNN, GRU, LSTMCell
from keras.layers import Conv2D, MaxPooling2D, Flatten, Bidirectional, LSTM, Dense, TimeDistributed, Conv1D, MaxPooling1D, Dropout, Conv3D
import warnings
warnings.filterwarnings('ignore')

---
<a id="6-1-arch"></a>
### 6.1 CNN Architecture

The architecture is defined as a `keras.Sequential` model. The encoder consists of 4 `Conv2D + MaxPooling2D` block pairs; the decoder consists of 4 `Conv2D + UpSampling2D` block pairs. A `Dropout(0.2)` layer sits at the bottleneck between encoder and decoder.

> 💡 **Commented-out `BatchNormalization` layers:** These are available as an optional extension. Batch normalisation can stabilise training and allow higher learning rates, but was not used in the benchmark results reported in the paper.

In [ ]:
model = keras.Sequential([
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    # keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.MaxPooling2D(),

    keras.layers.Dropout(0.2),

    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.UpSampling2D(),
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.UpSampling2D(),
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.UpSampling2D(),
    keras.layers.Conv2D(32, 5, padding='same', activation= 'swish'),
    keras.layers.UpSampling2D(),
    # keras.layers.BatchNormalization(),

    keras.layers.Conv2D(1, 5, padding='same'),

    # No activation since we are solving a regression problem
])

---
<a id="6-2-build"></a>
### 6.2 Build & Compile

The model is built with a batch of 32 samples to infer all layer shapes, then compiled with:
- **Optimiser:** Adam with learning rate $10^{-5}$ — a conservative initial rate suited to fine spatial prediction tasks.
- **Loss:** Mean Squared Error (MSE) — equivalent to minimising RMSE, the primary benchmark metric.

`model.summary()` prints the full layer-by-layer parameter count. The total trainable parameter count is the key number to note.

In [ ]:
model.build(X_train[:32].shape)
model.compile(keras.optimizers.Adam(learning_rate=1e-5), 'mse')
model.summary()

---
<a id="6-3-callbacks"></a>
### 6.3 Training Callbacks

Two callbacks control the training process:

**`ModelCheckpoint`** — saves the model weights to disk whenever the validation loss improves. This ensures the best-performing checkpoint is preserved even if later epochs overfit.

**`EarlyStopping`** (defined in the next cell) — halts training when validation loss stops improving for `patience` consecutive epochs, preventing wasted computation and overfitting.

> ⚠️ **Update the `filepath`** to a valid path on your system where the `.hdf5` checkpoint file should be saved.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

#create callback
filepath = 'D:/VSCODE_Works/BharatBench/ignore/IMDAA_CNN_H500_5days.hdf5'
checkpoint = ModelCheckpoint(filepath=filepath,
                             monitor='val_loss',
                             verbose=1,
                             save_best_only=True,
                             mode='min')

---
<a id="6-4-train"></a>
### 6.4 Training

The `fit_model` function wraps `model.fit` with the configured callbacks. Key training hyperparameters:

| Hyperparameter | Value | Notes |
|---|---|---|
| Epochs | 10 | Maximum; `EarlyStopping` will typically halt earlier |
| Batch size | 32 | Balances GPU utilisation and gradient noise |
| Shuffle | `False` | Temporal order is preserved to avoid leakage |
| Validation data | 2018 split | Drives `EarlyStopping` monitoring |
| Patience | 5 | Epochs without improvement before stopping |

> ⏱️ **Expected training time:** Approximately 5–15 minutes on CPU (Intel i7, 32 GB RAM) or 1–3 minutes on a GPU for 10 epochs at this grid resolution.  
> The paper reports a total training time of ~6 hours for the full multi-epoch training run used to produce the benchmark results.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 5, verbose=1)
def fit_model(model):

    history = model.fit(X_train, Y_train, epochs = 10,
                        validation_data= (X_valid, Y_valid) ,
                        batch_size = 32, shuffle = False,
                        callbacks = [early_stop])
    return history
history_cnn = fit_model(model)

---
<a id="6-5-save"></a>
### 6.5 Save Model & Plot Training History

**Save the trained model** to disk for later evaluation without retraining. The filename encodes key information (variable, lead time, best validation loss) — a useful convention for experiment tracking.

> ⚠️ **Update the save path** to match your directory structure.

In [ ]:
# save the model
model.save('D:/VSCODE_Works/BharatBench/ignore/IMDAA_CNN_T850_5days_val_loss_0.1473.hdf5')

**Plot training history** — the loss curves show training and validation MSE over epochs. Key patterns to look for:
- Both curves decreasing together → healthy training
- Validation loss plateauing while training loss continues to fall → overfitting onset (where `EarlyStopping` should have triggered)
- Large gap between training and validation loss → the model may benefit from stronger regularisation (e.g., higher dropout rate, more training data)

In [ ]:
# plot training history
# print("Values stored in history are ... \n", history_cnn.history)
plt.plot(history_cnn.history['loss'], label='train')
plt.plot(history_cnn.history['val_loss'], label='test')
plt.legend()
plt.show()

---
<a id="6-6-viz"></a>
### 6.6 Architecture Visualisation

Two options are provided for visualising the model architecture:

**Option A — `tf.keras.utils.plot_model`** (built-in, no extra install): Generates a layer graph diagram saved to PNG. Uncomment the line in the cell below to use it.

**Option B — `visualkeras`** (requires `pip install visualkeras`): Produces a 3D layered visualisation that more intuitively shows the spatial dimensions changing through the encoder-decoder. This is the option used in the paper's Figure 3.

In [ ]:
# Visualize the architecture of the CNN
# tf.keras.utils.plot_model(model, to_file= "CNN_IMDAA.png", show_shapes=True)

Install `visualkeras` if not already available in your environment:

In [ ]:
!pip install visualkeras

In [ ]:
import visualkeras

Generate a flat layered view of the CNN architecture and save to `output.png`:

In [ ]:
visualkeras.layered_view(model, to_file='output.png').show()

Generate an interactive 3D layered view with a legend showing layer types. `draw_volume=1` renders the spatial depth of each feature map:

In [ ]:
visualkeras.layered_view(model, legend=True, draw_volume = 1)

---
<a id="6-7-eval"></a>
### 6.7 Evaluation of the CNN Model

Load the best saved model checkpoint and evaluate on the test set (2019–2020).

> ⚠️ **Update the model path** to match the checkpoint saved in Section 6.5.

In [ ]:
from keras.models import Sequential, load_model
model = load_model('D:\VSCODE_Works\BharatBench\ignore\IMDAA_CNN_T850_5days_val_loss_0.1473.hdf5')

**Evaluation pipeline:**

1. **Generate predictions:** `model.predict(X_test)` runs the forward pass on normalised test inputs.
2. **Denormalise:** Predictions are in normalised units. They are converted back to physical units using the training-set statistics saved by the pipeline:
$$x_{\text{pred}} = \hat{y}_{\text{norm}} \times \sigma_{\text{train}} + \mu_{\text{train}}$$
3. **Wrap in xarray:** The predictions are assigned the correct time coordinates from the test dataset.
4. **Compute metrics:** RMSE, MAE, and ACC are computed against the un-normalised test observations.

> 🔁 **To evaluate a different variable**, update `std.TMP_prl` and `mean.TMP_prl` to match the variable you selected in Section 3.2 (e.g., `std.HGT_prl` and `mean.HGT_prl` for Z500).

In [ ]:
target = ds.sel(time=test_years)
# Convert predictions backto xarray
pred_test = X_test[:, :, :, 0].copy()
pred_test[:] = model.predict(X_test).squeeze()

# Unnormalize
pred_result = pred_test*std.TMP_prl.values + mean.TMP_prl.values
pred_result  = xr.DataArray(pred_result, dims=target.isel(time=slice(lead_time_steps, None)).dims, coords=target.isel(time=slice(lead_time_steps, None)).coords)
# compute RMSE
print('RMSE:', compute_rmse(pred_result, target.isel(time=slice(lead_time_steps, None))).TMP_prl.values)
print('MAE', compute_mae(pred_result, target.isel(time=slice(lead_time_steps, None))).TMP_prl.values)
print('ACC', compute_acc(pred_result, target.isel(time=slice(lead_time_steps, None))).TMP_prl.values)

---
<a id="7-convlstm"></a>
## Part B — ConvLSTM Encoder-Decoder Model

ConvLSTM (Convolutional Long Short-Term Memory) extends standard LSTM cells by replacing the matrix multiplications with convolutions. This makes it natively suited for **spatio-temporal sequences** — each hidden state is a feature map rather than a vector.

### ConvLSTM vs CNN — key differences

| Property | CNN | ConvLSTM |
|---|---|---|
| Input shape | `(batch, H, W, C)` | `(batch, T, H, W, C)` |
| Temporal modelling | None (single snapshot) | Explicit, via LSTM gates |
| Parameter count | Lower | Higher (gate matrices) |
| Suited for | Spatial pattern mapping | Spatio-temporal sequences |
| BharatBench result | Slightly better | Slightly worse (single-step input) |

> **Why does CNN outperform ConvLSTM in BharatBench?**  
> With a single initial time step as input (`T=1`), the LSTM gates have no historical sequence to exploit — the temporal memory provides no additional signal over a plain Conv2D. ConvLSTM's advantage would be expected to emerge with **multi-step inputs** (e.g., providing the past 4–8 time steps: $t_{-18h}, t_{-12h}, t_{-6h}, t_0$). This is highlighted as a direction for future work in the paper.

### ConvLSTM encoder-decoder architecture overview

```
Input (batch, T=1, 32, 32, 1)
│
├─ Encoder (×2 ConvLSTM2D + MaxPooling3D blocks)
│   ConvLSTM2D(32, 5×5, swish, return_sequences=True)
│   MaxPooling3D(1, 2, 2) — only spatial dims pooled
│   Spatial: 32 → 16 → 8
│
├─ Dropout(0.2)
│
├─ Decoder (×2 ConvLSTM2D + UpSampling3D blocks)
│   ConvLSTM2D(32, 5×5, swish, return_sequences=True)
│   UpSampling3D(1, 2, 2) — only spatial dims upsampled
│   Spatial: 8 → 16 → 32
│
└─ Output Conv3D(1, 5×5×5, linear)
   Shape: (batch, T=1, 32, 32, 1)
```

> **`MaxPooling3D(pool_size=(1, 2, 2))`:** The `1` in the temporal dimension means the temporal sequence length is preserved; only spatial resolution is halved. This is the 3D equivalent of `MaxPooling2D(2, 2)` from the CNN.

---
<a id="7-1-reshape"></a>
### 7.1 Reshape Data for ConvLSTM

ConvLSTM2D expects input of shape `(batch, timesteps, height, width, channels)`. The current arrays have shape `(batch, height, width, channels)` — the data pipeline was designed for the CNN.

`np.newaxis` inserts a new axis at position 1, converting shape `(N, 32, 32, 1)` → `(N, 1, 32, 32, 1)`. This represents a sequence of length 1 (the initial time step).

First, verify the current shapes before reshaping:

In [ ]:
print(X_train.shape)
print(Y_train.shape)
print(X_valid.shape)
print(Y_valid.shape)
print(X_test.shape)
print(Y_test.shape)

Add the sequence-length dimension (`T=1`) to all splits:

In [ ]:
X_train = X_train[:, np.newaxis,:,:,:]
Y_train = Y_train[:, np.newaxis,:,:,:]
X_valid = X_valid[:, np.newaxis,:,:,:]
Y_valid = Y_valid[:, np.newaxis,:,:,:]
X_test = X_test[:, np.newaxis,:,:,:]
Y_test = Y_test[:, np.newaxis,:,:,:]

---
<a id="7-2-arch"></a>
### 7.2 ConvLSTM Architecture

The ConvLSTM model uses `ConvLSTM2D` layers in place of `Conv2D`, and `MaxPooling3D` / `UpSampling3D` to handle the additional temporal dimension. The commented-out layers show the full 4-block version — the active configuration uses 2 encoder and 2 decoder blocks.

> 💡 **`return_sequences=True`** is required for all ConvLSTM layers that feed into another ConvLSTM layer or a pooling layer. It causes the layer to output the full sequence of hidden states rather than only the final state.

> 💡 **`Conv3D` output layer:** A 3D convolution is used at the output to process the `(T, H, W)` volume and produce the final prediction field. `kernel_size=(5, 5, 5)` with `padding='same'` preserves all three dimensions.

In [ ]:
# Build an convLSTM network
model = keras.Sequential([
    # keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    # keras.layers.MaxPooling3D(pool_size=(1, 2, 2)),
    # keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    # keras.layers.MaxPooling3D(pool_size=(1, 2, 2)),
    keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    keras.layers.MaxPooling3D(pool_size=(1, 2, 2)),
    keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    keras.layers.MaxPooling3D(pool_size=(1, 2, 2)),
    keras.layers.Dropout(0.2),
    keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    keras.layers.UpSampling3D(size=(1, 2, 2)),
    keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    keras.layers.UpSampling3D(size=(1, 2, 2)),
    # keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    # keras.layers.UpSampling3D(size=(1, 2, 2)),
    # keras.layers.ConvLSTM2D(filters=32, kernel_size=(5, 5), padding="same", return_sequences=True, activation= 'swish'),
    # keras.layers.UpSampling3D(size=(1, 2, 2)),
    
    keras.layers.Conv3D(filters=1, kernel_size=(5, 5, 5),  padding="same")    
])

---
<a id="7-3-build"></a>
### 7.3 Build & Compile

The model is built with explicit input shape `(None, 1, 32, 32, 1)` — `None` allows variable batch sizes at inference time. The Adam learning rate is set to $10^{-6}$, one order of magnitude lower than the CNN, reflecting ConvLSTM's greater sensitivity to learning rate during optimisation.

In [ ]:
model.build((None, 1, 32, 32,  1))
model.compile(keras.optimizers.Adam(learning_rate=1e-6), 'mse')
model.summary()

---
<a id="7-4-train"></a>
### 7.4 Callbacks & Training

The same checkpoint and early stopping strategy is used as for the CNN. Note that `patience=3` is used for the checkpoint callback (vs `patience=5` for the CNN), allowing slightly earlier stopping given ConvLSTM's tendency to overfit more quickly at this data scale.

> ⚠️ **Update the `filepath`** to a valid path on your system.

> ⏱️ **Expected training time:** ConvLSTM is significantly slower than CNN due to the recurrent gate computations. Expect 2–5× longer training time per epoch compared to the CNN at this grid size.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

#create callback
filepath = 'D:/VSCODE_Works/BharatBench/ignore/IMDAA_convlstm_H500_3days.hdf5'
checkpoint = ModelCheckpoint(filepath=filepath,
                             monitor='val_loss',
                             verbose=1,
                             save_best_only=True,
                             mode='min')

early_stop = keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 3, verbose=1)

The training loop is identical to the CNN — `fit_model` wraps `model.fit` with early stopping on validation loss. Maximum 5 epochs are specified; `EarlyStopping` (patience=5) will terminate early if validation loss plateaus.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 5, verbose=1)
def fit_model(model):

    history = model.fit(X_train, Y_train, epochs = 5,
                        validation_data= (X_valid, Y_valid) ,
                        batch_size = 32, shuffle = False,
                        callbacks = [early_stop])
    return history
history_cnn = fit_model(model)

---
<a id="7-5-save"></a>
### 7.5 Save & Load Model

Save the best-performing ConvLSTM checkpoint to disk:

> ⚠️ **Update the save path** to match your directory structure.

In [ ]:
model.save('D:/VSCODE_Works/BharatBench/ignore/IMDAA_convlstm_T850_5days_val_loss_0.1728.hdf5')

Load the saved model for evaluation. This cell can be run independently to evaluate a pre-trained model without re-running the training cells:

> ⚠️ **Update the load path** to match the path used when saving.

In [ ]:
from keras.models import Sequential, load_model
model = load_model('D:\VSCODE_Works\BharatBench\ignore\IMDAA_convlstm_T850_5days_val_loss_0.1728.hdf5')

---
<a id="7-6-eval"></a>
### 7.6 Evaluation of the ConvLSTM Model

**Evaluation pipeline** — identical to the CNN (Section 6.7), with one difference: `model.predict(X_test).squeeze()` removes the temporal dimension `T=1` that ConvLSTM adds to its output, restoring shape `(time, lat, lon)` before wrapping in xarray.

> 🔁 **To evaluate a different variable**, update `std.TMP_prl` and `mean.TMP_prl` to the variable selected in Section 3.2.

In [ ]:
target = ds.sel(time=test_years)

pred_test = model.predict(X_test).squeeze()

# Unnormalize
pred_result = pred_test*std.TMP_prl.values + mean.TMP_prl.values
pred_result  = xr.DataArray(pred_result, dims=target.isel(time=slice(lead_time_steps, None)).dims, coords=target.isel(time=slice(lead_time_steps, None)).coords)
# compute RMSE
print('RMSE:', compute_rmse(pred_result, target.isel(time=slice(lead_time_steps, None))).TMP_prl.values)
print('MAE', compute_mae(pred_result, target.isel(time=slice(lead_time_steps, None))).TMP_prl.values)
print('ACC', compute_acc(pred_result, target.isel(time=slice(lead_time_steps, None))).TMP_prl.values)

---
## Summary & Next Steps

This notebook has trained and evaluated both deep learning baselines for BharatBench.

### Results comparison table

Fill in your computed values after running the notebook:

| Model | Z500 RMSE (m) | T850 RMSE (K) | T2m RMSE (K) | Lead time |
|---|---|---|---|---|
| Persistence | — | — | — | 3-day |
| Climatology | — | — | — | — |
| Linear Regression | — | — | — | 3-day |
| **CNN** | — | — | — | 3-day |
| **CNN** | — | — | — | 5-day |
| **ConvLSTM** | — | — | — | 3-day |
| **ConvLSTM** | — | — | — | 5-day |

### Key observations from the paper

- The CNN achieves lower RMSE than linear regression for all variables at both lead times, confirming that non-linear spatial modelling adds value.
- ConvLSTM performs comparably to linear regression for Z500 and T850, and slightly better for T2m — suggesting the LSTM temporal memory provides limited benefit with single-step inputs.
- Precipitation (`APCP_sfc`) remains challenging for both architectures; a more sophisticated model (e.g., incorporating multi-level inputs, probabilistic outputs, or precipitation-specific loss functions) is needed.

### Directions for improvement

- **Multi-step input:** Feed the past 4–8 time steps as input to leverage ConvLSTM's temporal memory.
- **Multi-level input:** Use geopotential height at all pressure levels as input (as explored in Tables 5 & 6 of the paper).
- **Transfer learning:** Initialise 5-day models with weights from the 3-day trained model.
- **Probabilistic output:** Apply Monte Carlo Dropout at inference time (keep `Dropout` active during prediction) to generate ensemble forecasts and quantify uncertainty.

### Continuing with BharatBench

| Notebook | Description |
|---|---|
| `1_climatology_persistence.ipynb` | Non-learned baselines |
| `2_Linear_Regression.ipynb` | Linear regression baseline |
| `3_CNN_ConvLSTM.ipynb` | This notebook |

---
*BharatBench — MAS Lab, NIT Rourkela. Dataset: [Kaggle](https://www.kaggle.com/datasets/maslab/bharatbench) · Code: [GitHub](https://github.com/MASLABnitrkl/BharatBench)*